In [121]:
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver

import time
import random

# InMemorySaver saves the state into RAM. On closing the app, state will be erased.
# This is used for testing purpose. It is mostly not used in production.

In [122]:
load_dotenv()

True

In [123]:
model_gemini = ChatGoogleGenerativeAI(model = 'gemini-1.5-flash', temperature = 1)

In [124]:
random.randint(1,11)

1

In [125]:
# Defining the state

class CrashState(TypedDict):

    user_input: str
    step1: str
    step2: str
    step3: str

In [126]:
# Creating nodes

def step_1(state: CrashState) ->CrashState:

    print('Step_1 executed')
    return {"step1": 'Done'}

#----------------------------------------------------------------------------

def step_2(state: CrashState) ->CrashState:

    print('Step_2 hanging. Now manually interrupt the execution')
    time.sleep(10)
    print('Step_2 executed')
    return {"step2": 'Done'}

#---------------------------------------------------------------------------- 

def step_3(state: CrashState) -> CrashState:

    num = random.randint(1, 11)
    print('Step_3 executed')
    return {"step3": str(num)}

In [127]:
# Defining the graph
graph = StateGraph(CrashState)

# Adding nodes
graph.add_node('step1', step_1)
graph.add_node('step2', step_2)
graph.add_node('step3', step_3)

# Creating edges
graph.add_edge(START, 'step1')
graph.add_edge('step1', 'step2')
graph.add_edge('step2', 'step3')
graph.add_edge('step3', END)

# Creating a checkpointer's object
checkpointer = InMemorySaver()

# Compiling the graph
workflow = graph.compile(checkpointer = checkpointer)

In [117]:
initial_state = {"user_input": 'Start'}
config = {"configurable": {"thread_id": '1'}}

response = workflow.invoke(input = initial_state, config = config)

Step_1 executed
Step_2 hanging. Now manually interrupt the execution


KeyboardInterrupt: 

In [128]:
try:
    initial_state = {"user_input": 'Start'}
    config = {"configurable": {"thread_id": '1'}}
    print("▶️Execution started. Interrupt the session during step_2")
    response = workflow.invoke(input = {"user_input": "Start"}, config = config)
except KeyboardInterrupt:
    print("❌Kernel manually interrupted.")

▶️Execution started. Interrupt the session during step_2
Step_1 executed
Step_2 hanging. Now manually interrupt the execution
❌Kernel manually interrupted.


In [129]:
workflow.get_state(config = config)

StateSnapshot(values={'user_input': 'Start', 'step1': 'Done'}, next=('step2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f09175c-b271-6a12-8001-885992edb8c4'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2025-09-14T14:19:15.402395+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f09175c-b26f-6337-8000-59b24e850158'}}, tasks=(PregelTask(id='1e972a25-d464-9885-ee09-09c5ca6722e3', name='step2', path=('__pregel_pull', 'step2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [130]:
l = list(workflow.get_state_history(config = config))

for history in l:
    print(history)
    print('\n\n')

StateSnapshot(values={'user_input': 'Start', 'step1': 'Done'}, next=('step2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f09175c-b271-6a12-8001-885992edb8c4'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2025-09-14T14:19:15.402395+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f09175c-b26f-6337-8000-59b24e850158'}}, tasks=(PregelTask(id='1e972a25-d464-9885-ee09-09c5ca6722e3', name='step2', path=('__pregel_pull', 'step2'), error=None, interrupts=(), state=None, result=None),), interrupts=())



StateSnapshot(values={'user_input': 'Start'}, next=('step1',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f09175c-b26f-6337-8000-59b24e850158'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2025-09-14T14:19:15.401400+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f09175

# Notice:
* As we interrupted in step 2, see, only results till step 1 is added.
* See, `values={'user_input': 'Start', 'step1': 'Done'}, next=('step2',)`
* step2 and step3 is not yet added in the dictionary.
* Also, we can see, `next` is step2 as we interrupted there

**Now, let's resume from where it stoped. For this, we will have to pass the `thread_id` **

In [131]:
workflow.invoke(input = None, config = config)

# Note: It will start from Step_2
# See the final result. all three keys are assigned 'Done'

Step_2 hanging. Now manually interrupt the execution
Step_2 executed
Step_3 executed


{'user_input': 'Start', 'step1': 'Done', 'step2': 'Done', 'step3': '7'}

In [132]:
workflow.get_state(config = config)

StateSnapshot(values={'user_input': 'Start', 'step1': 'Done', 'step2': 'Done', 'step3': '7'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f091763-5948-6c57-8003-e981dd628ba0'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2025-09-14T14:22:13.958152+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f091763-5943-6e2e-8002-b63816e10778'}}, tasks=(), interrupts=())

In [133]:
l = list(workflow.get_state_history(config = config))

for history in l:
    print(history)
    print('\n\n')

StateSnapshot(values={'user_input': 'Start', 'step1': 'Done', 'step2': 'Done', 'step3': '7'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f091763-5948-6c57-8003-e981dd628ba0'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2025-09-14T14:22:13.958152+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f091763-5943-6e2e-8002-b63816e10778'}}, tasks=(), interrupts=())



StateSnapshot(values={'user_input': 'Start', 'step1': 'Done', 'step2': 'Done'}, next=('step3',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f091763-5943-6e2e-8002-b63816e10778'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-09-14T14:22:13.956151+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f09175c-b271-6a12-8001-885992edb8c4'}}, tasks=(PregelTask(id='72867e1b-338e-7041-3a66-60d2721bcb26', n

# 2) Human In The Loop(HIT)

Supose we are creating the workflow which generates a LinkedIn post and it posts on LinkedIn only after the user's approval.

This is called `Human In The Loop`. Unless user gives his approval, the workflow will not execute further.

**`We will learn this in a dedicated seassion`**

# 3) Time Travel (Used for debugging or Updating the state):

* Once execution of whole workflow is complete, `Time Travel` allows us to go back in any node and execute it from there.
* Supose a whole workflow executed and we wanna resume again from step 3, this is possible through `Time Travel`.
* We need to pass the particular checkpoint's id to reexecute from there.

**`Let' see how Time Travel Works**`

In [ ]:
# Re-executing step 3 of thread_id = '1'
# This will start the executing from step 3 and reach till END 

config = {"configurable": {"thread_id": '1', "checkpoint_id": '1f091763-5943-6e2e-8002-b63816e10778'}}

workflow.invoke(input = None, config = config)

Step_3 executed


{'user_input': 'Start', 'step1': 'Done', 'step2': 'Done', 'step3': '6'}

In [145]:
l = list(workflow.get_state_history(config = {"configurable": {"thread_id": '1'}}))

for history in l:
    print(history)
    print('\n\n')

# See step 3 has all the new values we got after re-executing.

StateSnapshot(values={'user_input': 'Start', 'step1': 'Done', 'step2': 'Done', 'step3': '6'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f091769-1b8e-65b8-8003-b4ca51ddfa9d'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2025-09-14T14:24:48.546757+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f091763-5943-6e2e-8002-b63816e10778'}}, tasks=(), interrupts=())



StateSnapshot(values={'user_input': 'Start', 'step1': 'Done', 'step2': 'Done', 'step3': '7'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f091769-070d-6a24-8003-d8a3cc4e1d64'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2025-09-14T14:24:46.396880+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f091763-5943-6e2e-8002-b63816e10778'}}, tasks=(), interrupts=())



StateSnapshot(values={'user_